# TEST — Bronze Ingest
Standalone test version of `bronze/ingest_file.ipynb`.  
All parameters are hardcoded below — just edit and **Run All**.

In [ ]:
# ============================================================
# TEST PARAMETERS — edit these to match your test scenario
# ============================================================
source_name   = "fedex_direct_invoice"
bronze_path   = "bronze/fedex/invoice/"
landing_path  = "landing/fedex/invoice/"
file_format   = "csv"          # csv | excel | parquet | json | avro

WORKSPACE   = "dev-fabric-data"
LAKEHOUSE   = "fabric_lakehouse"
DW_ENDPOINT = "8dac972c-0ff1-400f-bba3-e1e2302070b5.datawarehouse.fabric.microsoft.com"
DW_DATABASE = "fabric_gold_warehouse"

In [ ]:
import sys
from notebookutils import mssparkutils

FORMAT_GLOB = {
    "csv":     "*.csv",
    "excel":   "*.xlsx",
    "parquet": "*.parquet",
    "json":    "*.json",
    "avro":    "*.avro",
}
file_glob = FORMAT_GLOB.get(file_format, f"*.{file_format}")

### Shared lib & Config

In [ ]:
sys.path.insert(0, "/lakehouse/default/Files/shared")
import shared_functions as gf

run_id = "test-manual"

print(f"[TEST bronze/ingest] source={source_name}")
print(f"  landing : Files/{landing_path}")
print(f"  archive : Files/{bronze_path}")

### 1. List already-processed files

In [ ]:
with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
    records = gf.query_to_records(
        conn,
        f"SELECT file_name FROM {DW_DATABASE}.data_ops.pipeline_run_log "
        f"WHERE source_name = '{source_name}' AND layer = 'bronze' AND status = 'success'"
    )
    processed_files = {r["file_name"] for r in records}
print(f"  Already processed: {len(processed_files)} files")

### 2. List new files in landing zone

In [ ]:
landing_files = gf.list_files(
    workspace_name=WORKSPACE,
    lakehouse_name=LAKEHOUSE,
    prefix=landing_path,
    pattern=file_glob,
)

new_files = [f for f in landing_files if f.split("/")[-1] not in processed_files]
print(f"  Landing: {len(landing_files)} files, {len(new_files)} new")

if not new_files:
    print("  No new files — nothing to do. Stopping here.")

### 3. Move each new file: landing/ -> bronze/

In [ ]:
copied = []
errors = []

# Ensure destination directory exists (mssparkutils.fs.mv does not auto-create parents)
dest_dir_abfss = gf.onelake_abfss(
    WORKSPACE, LAKEHOUSE, f"Files/{bronze_path.rstrip('/')}"
)
mssparkutils.fs.mkdirs(dest_dir_abfss)  # noqa: F821

for src_abfss in new_files:
    file_name = src_abfss.split("/")[-1]
    dest_abfss = f"{dest_dir_abfss}/{file_name}"

    try:
        mssparkutils.fs.mv(src_abfss, dest_abfss, overwrite=True)  # noqa: F821
        print(f"  Moved {file_name} -> {dest_abfss}")
        copied.append(file_name)

        with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
            gf.log_run(
                conn,
                source_name=source_name,
                layer="bronze",
                status="success",
                rows_processed=1,
                file_name=file_name,
                run_id=run_id,
            )

    except Exception as exc:
        print(f"  Failed to move {file_name}: {exc}")
        errors.append({"file": file_name, "error": str(exc)})

print(f"\nResult: {len(copied)} copied, {len(errors)} errors")
if errors:
    print(f"Errors: {errors}")